<a href="https://colab.research.google.com/github/oihika/Spatial-Temporal-Storage-Optimization-for-Renewable-Power-Integration/blob/main/python%20code%20with%20output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# DASPI: Duration-Adaptive Storage Portfolio Intelligence
# ================================================================
# Research prototype (reviewer-revised; scenario results are not validated regional estimates) for:
# "Duration-Adaptive Storage Portfolio Intelligence (DASPI)
#  for High-Renewable-Penetration Power Systems"
#
# Author: Oihika Arpit
#
# IMPORTANT:
# This is a reproducible RESEARCH PROTOTYPE.
# Do not present generated results as final empirical findings
# until the assumptions, datasets and technology parameters
# are replaced/validated with cited sources.
#
# Main architecture:
#
# Coordinates
#     ↓
# NASA POWER renewable resource data
#     ↓
# Solar + Wind + Hydro + Demand
#     ↓
# Renewable-deficit signature
#     ↓
# Duration/severity analysis
#     ↓
# DMI
#     ↓
# Multi-technology optimisation
#     ↓
# Li-ion / Flow / PHS / H2 / Thermal
#     ↓
# DMI + RASV + TSCR
#     ↓
# Comparative + sensitivity analysis
#     ↓
# Publication figures + CSV results
# ================================================================


# ================================================================
# 0. INSTALL / IMPORT
# ================================================================

import sys
import subprocess
import importlib

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "requests": "requests"
}

for package, pip_name in required_packages.items():
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import math
import os
import warnings

from scipy.optimize import milp, LinearConstraint, Bounds

warnings.filterwarnings("default")

# ================================================================
# REVIEWER REVISION: REPRODUCIBILITY AND EXPLICIT ASSUMPTIONS
# ================================================================
from datetime import datetime, timezone
import hashlib

OUTPUT_DIR = os.environ.get("DASPI_OUTPUT_DIR", "/content/DASPI_Research_Outputs")
RAW_DIR = os.path.join(OUTPUT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(OUTPUT_DIR, "data", "processed")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
for _folder in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR):
    os.makedirs(_folder, exist_ok=True)

MODEL_LIFETIME_YEARS = 20.0
MAX_STORAGE_POWER_MW = 5000.0
MAX_STORAGE_ENERGY_MWH = 50000.0

# Explicit assumed resource-to-power conversion parameters. These are
# simplified screening assumptions, not a plant-specific PV/turbine model.
SOLAR_INPUT_UNIT_ASSUMPTION = "Treat NASA POWER hourly ALLSKY_SFC_SW_DWN as W/m^2; verify against archived response metadata"
PV_SYSTEM_DERATE = 0.80
PV_TEMPERATURE_COEFFICIENT_PER_C = -0.004
PV_REFERENCE_TEMPERATURE_C = 25.0
WIND_MEASUREMENT_HEIGHT_M = 10.0
WIND_HUB_HEIGHT_M = 100.0
WIND_SHEAR_EXPONENT = 0.14
WIND_TURBINE_MODEL = "generic normalized curve; no manufacturer-specific curve"

# Optional user-provided empirical inputs. If absent, constructed scenarios
# are used and must not be described as observed regional data.
DEMAND_CSV_PATH = os.environ.get("DASPI_DEMAND_CSV", "")
HYDRO_CSV_PATH = os.environ.get("DASPI_HYDRO_CSV", "")


print("DASPI environment ready.")


# ================================================================
# 1. USER CONFIGURATION
# ================================================================
#
# CHANGE THESE VALUES FOR YOUR CASE STUDY.
#
# Example:
# Jabalpur:     23.1815, 79.9864
# Bhopal:       23.2599, 77.4126
# Indore:       22.7196, 75.8577
#
# You can enter ANY location within valid geographic bounds.

LATITUDE = 23.1815
LONGITUDE = 79.9864

LOCATION_NAME = "Jabalpur, Madhya Pradesh, India"

# Coordinate constraint:
# Set this to True if you want the optimisation study restricted
# to a predefined geographical study region.

USE_COORDINATE_CONSTRAINT = True

# Example Central India boundary
LAT_MIN = 20.0
LAT_MAX = 25.5
LON_MIN = 74.0
LON_MAX = 82.5


# ================================================================
# TIME / DATA CONFIGURATION
# ================================================================

START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"

# 4-hour resolution is recommended for Colab prototype optimisation.
#
# For final research:
# RESAMPLE_HOURS = 1
#
# will provide hourly modelling but will require substantially
# more computational resources.

RESAMPLE_HOURS = 4

TIME_STEP = RESAMPLE_HOURS


# ================================================================
# RENEWABLE CONFIGURATION
# ================================================================

TARGET_RENEWABLE_SHARE = 0.70

# Solar / wind share of renewable capacity proxy
SOLAR_SHARE = 0.60
WIND_SHARE = 0.40

# Hydro is treated as an additional dispatchable renewable source.
HYDRO_CAPACITY_MW = 100.0

# Maximum renewable installed capacity allowed
MAX_SOLAR_MW = 5000.0
MAX_WIND_MW = 5000.0


# ================================================================
# DEFICIT DETECTION PARAMETERS
# ================================================================

# Deficit threshold:
# Positive residual demand means renewable generation
# is insufficient to meet demand.

DEFICIT_THRESHOLD_MW = 0.0

# Minimum duration of an event
MIN_DEFICIT_DURATION_HOURS = 4


# ================================================================
# STORAGE TECHNOLOGIES
# ================================================================
#
# Values below are MODEL ASSUMPTIONS for the prototype.
# Replace them with properly sourced values before publication.
#
# power_capex: USD/kW
# energy_capex: USD/kWh
# variable_cost: assumed USD/kWh
# eta_ch: charging efficiency
# eta_dis: discharging efficiency
# self_discharge: hourly fractional loss
# min_duration / max_duration: technology operating range
#
# Duration is NOT imposed as the final answer.
# The optimisation chooses E/P inside the technology's
# feasible operating envelope.

STORAGE = {

    "Li-ion": {
        "power_capex": 300,
        "energy_capex": 180,
        "eta_ch": 0.95,
        "eta_dis": 0.95,
        "self_discharge": 0.0001,
        "min_duration": 1,
        "max_duration": 8,
        "variable_cost": 0.002
    },

    "Flow Battery": {
        "power_capex": 500,
        "energy_capex": 100,
        "eta_ch": 0.88,
        "eta_dis": 0.88,
        "self_discharge": 0.00005,
        "min_duration": 4,
        "max_duration": 24,
        "variable_cost": 0.003
    },

    "Pumped Hydro": {
        "power_capex": 1200,
        "energy_capex": 40,
        "eta_ch": 0.85,
        "eta_dis": 0.90,
        "self_discharge": 0.00001,
        "min_duration": 6,
        "max_duration": 100,
        "variable_cost": 0.001
    },

    "Hydrogen": {
        "power_capex": 900,
        "energy_capex": 15,
        "eta_ch": 0.65,
        "eta_dis": 0.55,
        "self_discharge": 0.000001,
        "min_duration": 12,
        "max_duration": 1000,
        "variable_cost": 0.010
    },

    "Thermal Storage": {
        "power_capex": 400,
        "energy_capex": 50,
        "eta_ch": 0.80,
        "eta_dis": 0.80,
        "self_discharge": 0.001,
        "min_duration": 4,
        "max_duration": 72,
        "variable_cost": 0.004
    }
}

TECHNOLOGIES = list(STORAGE.keys())
N_TECH = len(TECHNOLOGIES)


# ================================================================
# 2. COORDINATE VALIDATION
# ================================================================

def validate_coordinates(lat, lon):

    if not (-90 <= lat <= 90):
        raise ValueError("Latitude must be between -90 and +90 degrees.")

    if not (-180 <= lon <= 180):
        raise ValueError("Longitude must be between -180 and +180 degrees.")

    if USE_COORDINATE_CONSTRAINT:

        inside = (
            LAT_MIN <= lat <= LAT_MAX
            and
            LON_MIN <= lon <= LON_MAX
        )

        if not inside:
            raise ValueError(
                f"""
COORDINATE CONSTRAINT VIOLATION

Selected coordinate:
Latitude  = {lat}
Longitude = {lon}

Allowed study region:
Latitude  = {LAT_MIN} to {LAT_MAX}
Longitude = {LON_MIN} to {LON_MAX}

Change the coordinates or disable USE_COORDINATE_CONSTRAINT.
"""
            )

    print("\nCoordinate validation successful.")
    print(f"Latitude  : {lat:.6f}")
    print(f"Longitude : {lon:.6f}")
    print(f"Location  : {LOCATION_NAME}")

    return True


validate_coordinates(LATITUDE, LONGITUDE)


# ================================================================
# 3. NASA POWER DATA DOWNLOAD
# ================================================================

def get_nasa_power_data(lat, lon, start_date, end_date):
    """Fetch NASA POWER hourly data and archive the exact request and raw response."""
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN,WS10M,T2M",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start_date.replace("-", ""),
        "end": end_date.replace("-", ""),
        "format": "JSON",
        "time-standard": "UTC"
    }
    retrieved_utc = datetime.now(timezone.utc).isoformat()
    print("\nDownloading NASA POWER renewable-resource data...")
    response = requests.get(url, params=params, timeout=90)
    response.raise_for_status()
    payload = response.json()
    # Preserve raw response immediately, even if a later unit-validation check fails.
    with open(os.path.join(RAW_DIR, "nasa_power_hourly_raw.json"), "w", encoding="utf-8") as f:
        json.dump(payload, f)

    with open(os.path.join(RAW_DIR, "nasa_power_request_metadata.json"), "w", encoding="utf-8") as f:
        json.dump({
            "requested_url": url, "resolved_url": response.url,
            "parameters": params, "retrieved_at_utc": retrieved_utc,
            "http_status": response.status_code,
            "response_sha256": hashlib.sha256(response.content).hexdigest(),
            "validation_status": "pending unit validation"
        }, f, indent=2)
    parameter_metadata = payload.get("parameters", {})
    response_units = {
        key: parameter_metadata.get(key, {}).get("units")
        for key in ("ALLSKY_SFC_SW_DWN", "WS10M", "T2M")
    }
    solar_unit = str(response_units.get("ALLSKY_SFC_SW_DWN", "")).lower().replace(" ", "")
    if not ("w/m" in solar_unit or "watt" in solar_unit):
        raise ValueError(
            "NASA POWER solar units were not verified as irradiance. "
            f"Response units={response_units}. Do not run PV conversion until units are resolved."
        )
    wind_unit = str(response_units.get("WS10M", "")).lower().replace(" ", "")
    if not ("m/s" in wind_unit or "m.s-1" in wind_unit or "msec-1" in wind_unit):
        raise ValueError(f"NASA POWER wind units were not verified as m/s: {response_units}")

    request_record = {
        "requested_url": url,
        "resolved_url": response.url,
        "parameters": params,
        "retrieved_at_utc": retrieved_utc,
        "http_status": response.status_code,
        "response_sha256": hashlib.sha256(response.content).hexdigest(),
        "response_header": payload.get("header", {}),
        "response_parameter_metadata": parameter_metadata,
        "verified_response_units": response_units,
        "solar_unit_assumption": SOLAR_INPUT_UNIT_ASSUMPTION,
        "wind_measurement_height_m": WIND_MEASUREMENT_HEIGHT_M,
        "wind_hub_height_m": WIND_HUB_HEIGHT_M,
        "wind_shear_exponent_assumption": WIND_SHEAR_EXPONENT,
        "pv_system_derate_assumption": PV_SYSTEM_DERATE,
        "pv_temperature_coefficient_per_C_assumption": PV_TEMPERATURE_COEFFICIENT_PER_C
    }
    with open(os.path.join(RAW_DIR, "nasa_power_request_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(request_record, f, indent=2)
    with open(os.path.join(RAW_DIR, "nasa_power_hourly_raw.json"), "w", encoding="utf-8") as f:
        json.dump(payload, f)

    properties = payload["properties"]["parameter"]
    solar = properties["ALLSKY_SFC_SW_DWN"]
    wind = properties["WS10M"]
    temp = properties["T2M"]
    rows = []
    for timestamp in solar.keys():
        rows.append({
            "datetime": pd.to_datetime(timestamp, format="%Y%m%d%H", utc=True),
            "solar_resource": solar[timestamp],
            "wind_speed_10m": wind.get(timestamp, np.nan),
            "temperature_C": temp.get(timestamp, np.nan)
        })
    out = pd.DataFrame(rows).sort_values("datetime").set_index("datetime")
    out = out.replace([-999, -999.0], np.nan)
    missing_before = out.isna().sum().to_dict()
    # Interpolate at most two consecutive missing hourly values; raw JSON is preserved.
    out = out.interpolate(limit=2, limit_area="inside")
    missing_after = out.isna().sum().to_dict()
    out = out.dropna()
    expected = (pd.Timestamp(end_date, tz="UTC") - pd.Timestamp(start_date, tz="UTC")).days * 24 + 24
    print(f"Valid hourly observations: {len(out):,}; expected approximately {expected:,}.")
    print("Missing counts before limited interpolation:", missing_before)
    print("Missing counts after limited interpolation:", missing_after)
    if len(out) == 0:
        raise ValueError("NASA POWER returned no valid rows; inspect raw response and units.")
    out.to_csv(os.path.join(PROCESSED_DIR, "nasa_power_hourly_processed.csv"))
    return out


resource = get_nasa_power_data(
    LATITUDE,
    LONGITUDE,
    START_DATE,
    END_DATE
)


# ================================================================
# 4. RENEWABLE RESOURCE CHECK
# ================================================================

print("\n================================================")
print("LOCATION RESOURCE DIAGNOSTIC")
print("================================================")

print(
    f"Mean solar resource indicator : "
    f"{resource['solar_resource'].mean():.3f}"
)

print(
    f"Mean wind speed               : "
    f"{resource['wind_speed_10m'].mean():.3f} m/s"
)

print(
    f"Maximum wind speed            : "
    f"{resource['wind_speed_10m'].max():.3f} m/s"
)

print(
    f"Mean temperature              : "
    f"{resource['temperature'].mean():.2f} °C"
)


# ================================================================
# 5. WIND POWER CURVE
# ================================================================

def wind_power_curve(v_hub_m_s):
    """Generic normalized curve: cut-in 3, rated 12, cut-out 25 m/s."""
    v = np.asarray(v_hub_m_s, dtype=float)
    power = np.zeros_like(v, dtype=float)
    mask1 = (v >= 3.0) & (v < 12.0)
    power[mask1] = (v[mask1]**3 - 3.0**3) / (12.0**3 - 3.0**3)
    mask2 = (v >= 12.0) & (v <= 25.0)
    power[mask2] = 1.0
    return np.clip(power, 0.0, 1.0)

# Explicit power-law adjustment from 10 m to assumed 100 m hub height.
# The shear exponent is an assumption; replace with site-specific evidence.
resource["wind_speed_hub_m_s"] = (
    resource["wind_speed_10m"]
    * (WIND_HUB_HEIGHT_M / WIND_MEASUREMENT_HEIGHT_M) ** WIND_SHEAR_EXPONENT
)
resource["wind_speed"] = resource["wind_speed_10m"]  # compatibility alias for reporting
resource["wind_cf"] = wind_power_curve(resource["wind_speed_hub_m_s"].values)


# ================================================================
# 6. SOLAR NORMALISATION
# ================================================================

# Simplified resource-to-PV conversion with explicit, auditable assumptions.
# Verify the NASA POWER parameter units from the archived raw response metadata.
# This is not a plant-specific PV model and must not be described as PVsyst.
solar_irradiance_w_m2 = resource["solar_resource"].clip(lower=0)
temp_factor = 1.0 + PV_TEMPERATURE_COEFFICIENT_PER_C * (
    resource["temperature_C"] - PV_REFERENCE_TEMPERATURE_C
)
temp_factor = temp_factor.clip(lower=0.0, upper=1.2)
resource["solar_cf"] = (
    (solar_irradiance_w_m2 / 1000.0) * PV_SYSTEM_DERATE * temp_factor
).clip(lower=0.0, upper=1.0)
if resource["solar_cf"].max() <= 0:
    raise ValueError("Solar resource data has no usable values; verify NASA POWER units.")


# ================================================================
# 7. RESAMPLE DATA
# ================================================================

df = resource.resample(
    f"{RESAMPLE_HOURS}h"
).mean()

df = df.dropna()


# ================================================================
# 8. REPRESENTATIVE ELECTRICITY DEMAND
# ================================================================
#
# IMPORTANT:
# This is a transparent modelling profile, NOT measured electricity
# demand. Replace it with actual Central India demand data for
# publication.
#
# The profile contains:
# - daily cycle
# - evening peak
# - seasonal variation
# - weekday/weekend effect

def create_demand_profile(index):

    hour = index.hour.values
    dayofyear = index.dayofyear.values
    weekday = index.dayofweek.values

    daily_component = (
        0.65
        + 0.15 * np.sin(
            2 * np.pi * (hour - 7) / 24
        )
        + 0.20 * np.exp(
            -((hour - 20) / 3.0)**2
        )
    )

    seasonal_component = (
        1.0
        + 0.10 * np.cos(
            2 * np.pi * (dayofyear - 200) / 365
        )
    )

    weekend_component = np.where(
        weekday >= 5,
        0.92,
        1.0
    )

    demand = (
        daily_component
        * seasonal_component
        * weekend_component
    )

    # Normalise around 500 MW mean demand
    demand = (
        demand
        /
        demand.mean()
        *
        500.0
    )

    return demand


if DEMAND_CSV_PATH and os.path.isfile(DEMAND_CSV_PATH):
    _demand = pd.read_csv(DEMAND_CSV_PATH, parse_dates=["datetime"])
    if not {"datetime", "demand_MW"}.issubset(_demand.columns):
        raise ValueError("Demand CSV must contain datetime and demand_MW columns.")
    _demand["datetime"] = pd.to_datetime(_demand["datetime"], utc=True)
    _demand = _demand.set_index("datetime").sort_index()
    df = df.join(_demand[["demand_MW"]], how="inner")
    DEMAND_DATA_STATUS = "external CSV supplied; document source, units, and preprocessing"
else:
    df["demand_MW"] = create_demand_profile(df.index)
    DEMAND_DATA_STATUS = "synthetic constructed profile normalised to 500 MW mean; not empirical"
print("Demand data status:", DEMAND_DATA_STATUS)


# ================================================================
# 9. HYDRO PROFILE
# ================================================================
#
# Placeholder hydro availability profile.
#
# Replace this with actual reservoir / river-flow / hydro dispatch
# data in the final study.

month = df.index.month.values

hydro_monthly_cf = {
    1: 0.45,
    2: 0.42,
    3: 0.40,
    4: 0.38,
    5: 0.40,
    6: 0.55,
    7: 0.80,
    8: 0.85,
    9: 0.78,
    10: 0.65,
    11: 0.55,
    12: 0.48
}

df["hydro_cf"] = [
    hydro_monthly_cf[m]
    for m in month
]

df["hydro_MW"] = HYDRO_CAPACITY_MW * df["hydro_cf"]
if HYDRO_CSV_PATH and os.path.isfile(HYDRO_CSV_PATH):
    _hydro = pd.read_csv(HYDRO_CSV_PATH, parse_dates=["datetime"])
    if not {"datetime", "hydro_MW"}.issubset(_hydro.columns):
        raise ValueError("Hydro CSV must contain datetime and hydro_MW columns.")
    _hydro["datetime"] = pd.to_datetime(_hydro["datetime"], utc=True)
    _hydro = _hydro.set_index("datetime").sort_index()
    df = df.drop(columns=["hydro_MW"]).join(_hydro[["hydro_MW"]], how="inner")
    df["hydro_cf"] = (df["hydro_MW"] / HYDRO_CAPACITY_MW).clip(0, 1)
    HYDRO_DATA_STATUS = "external CSV supplied; document source, units, and preprocessing"
else:
    HYDRO_DATA_STATUS = "synthetic monthly capacity-factor placeholder; not observed hydropower"
print("Hydropower data status:", HYDRO_DATA_STATUS)
df.to_csv(os.path.join(PROCESSED_DIR, "model_timeseries_before_capacity_calibration.csv"))


# ================================================================
# 10. INITIAL RENEWABLE CAPACITY CALIBRATION
# ================================================================
#
# We calculate renewable capacities to approximately reach the
# target renewable-energy contribution.
#
# Solar and wind are calibrated using the observed resource profiles.

annual_load_MWh = (
    df["demand_MW"].sum()
    * RESAMPLE_HOURS
)

annual_hydro_MWh = (
    df["hydro_MW"].sum()
    * RESAMPLE_HOURS
)

target_RE_MWh = (
    TARGET_RENEWABLE_SHARE
    * annual_load_MWh
)

remaining_RE_MWh = max(
    target_RE_MWh - annual_hydro_MWh,
    0
)

solar_cf_mean = df["solar_cf"].mean()
wind_cf_mean = df["wind_cf"].mean()

# Allocate the remaining renewable energy between solar and wind.
solar_energy_target = remaining_RE_MWh * SOLAR_SHARE
wind_energy_target = remaining_RE_MWh * WIND_SHARE

solar_capacity = (
    solar_energy_target
    /
    (
        solar_cf_mean
        * len(df) * RESAMPLE_HOURS
    )
)

wind_capacity = (
    wind_energy_target
    /
    (
        wind_cf_mean
        * len(df) * RESAMPLE_HOURS
    )
)

solar_capacity = min(
    solar_capacity,
    MAX_SOLAR_MW
)

wind_capacity = min(
    wind_capacity,
    MAX_WIND_MW
)

df["solar_MW"] = (
    df["solar_cf"]
    * solar_capacity
)

df["wind_MW"] = (
    df["wind_cf"]
    * wind_capacity
)

df["renewable_available_MW"] = (
    df["solar_MW"]
    +
    df["wind_MW"]
    +
    df["hydro_MW"]
)

df["net_load_MW"] = (
    df["demand_MW"]
    -
    df["renewable_available_MW"]
)
df.to_csv(os.path.join(PROCESSED_DIR, "model_timeseries_calibrated.csv"))

print("\n================================================")
print("SYSTEM CALIBRATION")
print("================================================")

print(
    f"Solar capacity       : {solar_capacity:.2f} MW"
)

print(
    f"Wind capacity        : {wind_capacity:.2f} MW"
)

print(
    f"Hydro capacity       : {HYDRO_CAPACITY_MW:.2f} MW"
)

print(
    f"Annual load          : {annual_load_MWh:,.0f} MWh"
)


# ================================================================
# 11. RENEWABLE DEFICIT SIGNATURE
# ================================================================

df["deficit_MW"] = np.maximum(
    df["demand_MW"]
    -
    df["renewable_available_MW"],
    0
)

df["surplus_MW"] = np.maximum(
    df["renewable_available_MW"]
    -
    df["demand_MW"],
    0
)


# ================================================================
# 12. DEFICIT EPISODE DETECTION
# ================================================================

def detect_deficit_events(
    deficit_series,
    timestep_hours=1,
    threshold=0
):

    events = []

    active = False
    start = None

    values = deficit_series.values
    times = deficit_series.index

    for i, value in enumerate(values):

        is_deficit = value > threshold

        if is_deficit and not active:

            active = True
            start = i

        elif not is_deficit and active:

            end = i - 1

            duration = (
                (end - start + 1)
                * timestep_hours
            )

            if duration >= MIN_DEFICIT_DURATION_HOURS:

                segment = values[start:end+1]

                events.append({
                    "start": times[start],
                    "end": times[end],
                    "duration_h": duration,
                    "energy_deficit_MWh":
                        segment.sum() * timestep_hours,
                    "peak_deficit_MW":
                        segment.max(),
                    "mean_deficit_MW":
                        segment.mean()
                })

            active = False

    # Handle event reaching end
    if active:

        end = len(values) - 1

        duration = (
            (end - start + 1)
            * timestep_hours
        )

        if duration >= MIN_DEFICIT_DURATION_HOURS:

            segment = values[start:end+1]

            events.append({
                "start": times[start],
                "end": times[end],
                "duration_h": duration,
                "energy_deficit_MWh":
                    segment.sum() * timestep_hours,
                "peak_deficit_MW":
                    segment.max(),
                "mean_deficit_MW":
                    segment.mean()
            })

    return pd.DataFrame(events)


events = detect_deficit_events(
    df["deficit_MW"],
    RESAMPLE_HOURS,
    DEFICIT_THRESHOLD_MW
)

print("\n================================================")
print("DEFICIT SIGNATURE")
print("================================================")

print(
    f"Number of deficit events : {len(events)}"
)

if len(events) > 0:

    print(
        f"Longest deficit         : "
        f"{events['duration_h'].max():.1f} h"
    )

    print(
        f"Mean deficit duration   : "
        f"{events['duration_h'].mean():.1f} h"
    )

    print(
        f"Maximum deficit power   : "
        f"{events['peak_deficit_MW'].max():.2f} MW"
    )

    print(
        f"Total deficit energy    : "
        f"{events['energy_deficit_MWh'].sum():,.0f} MWh"
    )


# ================================================================
# 13. DURATION MISMATCH INDEX (DMI)
# ================================================================
#
# DMI asks:
#
# How strongly does the observed renewable-deficit structure
# exceed the effective duration envelope of a short-duration
# storage technology?
#
# Here the reference duration is 8 hours.
#
# DMI > 1 means significant exposure to events beyond the
# reference short-duration envelope.

REFERENCE_DURATION_HOURS = 8.0


def calculate_DMI(events):

    if len(events) == 0:
        return 0.0

    excess = np.maximum(
        events["duration_h"]
        -
        REFERENCE_DURATION_HOURS,
        0
    )

    weighted_excess = (
        excess
        *
        events["energy_deficit_MWh"]
    ).sum()

    total_energy = (
        events["energy_deficit_MWh"].sum()
    )

    if total_energy == 0:
        return 0.0

    dmi = (
        weighted_excess
        /
        (
            REFERENCE_DURATION_HOURS
            *
            total_energy
        )
    )

    return float(dmi)


DMI = calculate_DMI(events)
events.to_csv(os.path.join(RESULTS_DIR, "deficit_event_table.csv"), index=False)

# Formula-level regression test using a hand-calculated synthetic event table.
_dmi_test = pd.DataFrame({
    "duration_h": [16.0, 4.0],
    "energy_deficit_MWh": [100.0, 100.0]
})
_dmi_test_value = calculate_DMI(_dmi_test)
assert np.isclose(_dmi_test_value, 0.5), f"DMI unit test failed: {_dmi_test_value}"
print("DMI formula unit test passed (synthetic test only):", _dmi_test_value)

print(
    f"\nDuration Mismatch Index (DMI): {DMI:.4f}"
)


# ================================================================
# 14. MILP MODEL
# ================================================================
#
# The optimisation determines:
#
#   Pcap[k] = storage power capacity
#   Ecap[k] = storage energy capacity
#   charge[k,t]
#   discharge[k,t]
#   SOC[k,t]
#   renewable_used[t]
#   grid[t]
#   unmet[t]
#
# Objective:
#
# minimise
#
# annualised storage cost
# + grid energy cost
# + unmet-load penalty
# + storage throughput cost
#
# Subject to:
#
# energy balance
# SOC dynamics
# capacity limits
# duration constraints
# renewable availability
#
# ================================================================

def build_and_solve_DASPI(
    data,
    technologies,
    storage_parameters,
    fixed_energy_shares=None,
    spatial_config=None,
    grid_cost_override=None
):

    N = len(data)
    K = len(technologies)

    dt = RESAMPLE_HOURS

    # ------------------------------------------------------------
    # VARIABLE INDEXING
    # ------------------------------------------------------------

    # Arrays:
    #
    # charge       K x N
    # discharge    K x N
    # SOC          K x N
    # renewable    N
    # grid         N
    # unmet        N
    #
    # capacities:
    #
    # Pcap         K
    # Ecap         K

    offset = 0

    charge_idx = np.arange(
        offset,
        offset + K*N
    ).reshape(K, N)

    offset += K*N

    discharge_idx = np.arange(
        offset,
        offset + K*N
    ).reshape(K, N)

    offset += K*N

    soc_idx = np.arange(
        offset,
        offset + K*N
    ).reshape(K, N)

    offset += K*N

    renewable_used_idx = np.arange(
        offset,
        offset + N
    )

    offset += N

    grid_idx = np.arange(
        offset,
        offset + N
    )

    offset += N

    unmet_idx = np.arange(
        offset,
        offset + N
    )

    offset += N

    Pcap_idx = np.arange(
        offset,
        offset + K
    )

    offset += K

    Ecap_idx = np.arange(
        offset,
        offset + K
    )

    offset += K

    # Binary technology-installation decisions make this a genuine MILP.
    install_idx = np.arange(offset, offset + K)
    offset += K

    nvars = offset

    print(
        f"\nBuilding DASPI MILP with {nvars:,} variables..."
    )

    # ------------------------------------------------------------
    # OBJECTIVE
    # ------------------------------------------------------------

    c = np.zeros(nvars)
    horizon_fraction_of_year = (N * dt) / 8760.0

    # Capital costs are annualised and prorated to the modeled horizon.
    # For the full 8760-hour annual run, horizon_fraction_of_year = 1.
    for k, tech in enumerate(technologies):

        p = storage_parameters[tech]

        # Optional spatial cost proxy. The multiplier is a transparent
        # scenario assumption representing an equivalent connection-cost
        # uplift; it is NOT an observed interconnection tariff.
        connection_multiplier = 1.0
        connection_cost_usd_per_mw_km = 0.0
        distance_km = 0.0
        if spatial_config is not None:
            connection_multiplier = float(
                spatial_config.get("connection_cost_multiplier", {}).get(tech, 1.0)
            )
            distance_km = float(spatial_config.get("distance_km", {}).get(tech, 0.0))
            connection_cost_usd_per_mw_km = float(
                spatial_config.get("connection_cost_usd_per_mw_km", 0.0)
            )
            if connection_multiplier < 1.0 or distance_km < 0 or connection_cost_usd_per_mw_km < 0:
                raise ValueError("Spatial cost multipliers, distances, and cost rates must be non-negative (multiplier >= 1).")

        c[Pcap_idx[k]] = (
            (p["power_capex"] * 1000.0 * connection_multiplier
             + distance_km * connection_cost_usd_per_mw_km)
            / MODEL_LIFETIME_YEARS
            * horizon_fraction_of_year
        )

        c[Ecap_idx[k]] = (
            p["energy_capex"] * 1000.0
            / MODEL_LIFETIME_YEARS
            * horizon_fraction_of_year
        )

        # Variable storage throughput cost
        c[
            charge_idx[k, :]
        ] = (
            p["variable_cost"]
            * 1000.0
            * dt
        )

        c[
            discharge_idx[k, :]
        ] = (
            p["variable_cost"]
            * 1000.0
            * dt
        )

    # Grid purchase tariff in USD/MWh. The default 150 USD/MWh is an
    # illustrative assumption equivalent to 0.15 USD/kWh; replace with a
    # documented tariff for the study region.
    GRID_COST = 150.0 if grid_cost_override is None else float(grid_cost_override)
    if GRID_COST < 0:
        raise ValueError("Grid purchase cost cannot be negative.")

    c[grid_idx] = (
        GRID_COST * dt
    )

    # Unserved-energy penalty in USD/MWh; explicit illustrative assumption.
    VALUE_OF_LOST_LOAD = 10000.0

    c[unmet_idx] = (
        VALUE_OF_LOST_LOAD * dt
    )

    # ------------------------------------------------------------
    # BOUNDS
    # ------------------------------------------------------------

    lower = np.zeros(nvars)
    upper = np.full(nvars, np.inf)
    upper[Pcap_idx] = MAX_STORAGE_POWER_MW
    upper[Ecap_idx] = MAX_STORAGE_ENERGY_MWH
    upper[install_idx] = 1.0

    # Optional hard spatial eligibility constraints. A technology mapped to
    # an ineligible candidate site is prohibited by setting both installed
    # power and energy capacity to zero. Eligibility data in the demonstration
    # below are synthetic and must not be interpreted as real Jabalpur site data.
    if spatial_config is not None:
        eligible = spatial_config.get("eligible", {})
        max_power_by_tech = spatial_config.get("max_power_MW", {})
        max_energy_by_tech = spatial_config.get("max_energy_MWh", {})
        for k, tech in enumerate(technologies):
            if eligible.get(tech, True) is False:
                upper[Pcap_idx[k]] = 0.0
                upper[Ecap_idx[k]] = 0.0
            else:
                upper[Pcap_idx[k]] = min(upper[Pcap_idx[k]], float(max_power_by_tech.get(tech, MAX_STORAGE_POWER_MW)))
                upper[Ecap_idx[k]] = min(upper[Ecap_idx[k]], float(max_energy_by_tech.get(tech, MAX_STORAGE_ENERGY_MWH)))

    # Renewable used <= available
    upper[renewable_used_idx] = (
        data["renewable_available_MW"].values
    )

    # Grid and unmet are bounded for numerical stability
    upper[grid_idx] = data["demand_MW"].values * 2
    if spatial_config is not None and "grid_import_limit_MW" in spatial_config:
        grid_limit = float(spatial_config["grid_import_limit_MW"])
        if grid_limit < 0:
            raise ValueError("Spatial grid-import limit must be non-negative.")
        upper[grid_idx] = np.minimum(upper[grid_idx], grid_limit)

    upper[unmet_idx] = (
        data["demand_MW"].values
    )

    # ------------------------------------------------------------
    # CONSTRAINT MATRICES
    # ------------------------------------------------------------

    A = []
    lb = []
    ub = []

    # ------------------------------------------------------------
    # ENERGY BALANCE
    # ------------------------------------------------------------

    for t in range(N):

        row = {}

        # renewable used
        row[renewable_used_idx[t]] = 1.0

        # grid
        row[grid_idx[t]] = 1.0

        # discharge
        for k in range(K):

            row[
                discharge_idx[k, t]
            ] = row.get(
                discharge_idx[k, t],
                0
            ) + 1.0

        # charge
        for k in range(K):

            row[
                charge_idx[k, t]
            ] = row.get(
                charge_idx[k, t],
                0
            ) - 1.0

        # unmet demand
        row[unmet_idx[t]] = 1.0

        # Equation:
        #
        # renewable + grid + discharge + unmet
        # =
        # demand + charge
        #
        # => renewable + grid + discharge + unmet - charge
        # = demand

        A.append(row)
        lb.append(
            data["demand_MW"].iloc[t]
        )
        ub.append(
            data["demand_MW"].iloc[t]
        )

    # ------------------------------------------------------------
    # STORAGE SOC EQUATIONS
    # ------------------------------------------------------------

    for k, tech in enumerate(technologies):

        p = storage_parameters[tech]

        eta_ch = p["eta_ch"]
        eta_dis = p["eta_dis"]

        decay = (
            1
            -
            p["self_discharge"]
        ) ** dt

        for t in range(N):

            row = {}

            # SOC_t
            row[soc_idx[k, t]] = 1.0

            # charge contribution
            row[
                charge_idx[k, t]
            ] = -eta_ch * dt

            # discharge contribution
            row[
                discharge_idx[k, t]
            ] = (
                dt / eta_dis
            )

            if t == 0:

                # Initial SOC = 50% of capacity
                row[Ecap_idx[k]] = -0.50

                # SOC_0 - charge + discharge - 0.5 Ecap = 0
                rhs = 0.0

            else:

                # Previous SOC
                row[
                    soc_idx[k, t-1]
                ] = -decay

                rhs = 0.0

            A.append(row)
            lb.append(rhs)
            ub.append(rhs)

        # Cyclic terminal condition: end-of-horizon SOC returns to 50% capacity.
        terminal_row = {soc_idx[k, N-1]: 1.0, Ecap_idx[k]: -0.50}
        A.append(terminal_row)
        lb.append(0.0)
        ub.append(0.0)

    # ------------------------------------------------------------
    # STORAGE POWER CAPACITY
    # ------------------------------------------------------------

    for k in range(K):

        for t in range(N):

            # charge <= Pcap
            row = {
                charge_idx[k, t]: 1.0,
                Pcap_idx[k]: -1.0
            }

            A.append(row)
            lb.append(-np.inf)
            ub.append(0.0)

            # discharge <= Pcap
            row = {
                discharge_idx[k, t]: 1.0,
                Pcap_idx[k]: -1.0
            }

            A.append(row)
            lb.append(-np.inf)
            ub.append(0.0)

            # SOC <= Ecap
            row = {
                soc_idx[k, t]: 1.0,
                Ecap_idx[k]: -1.0
            }

            A.append(row)
            lb.append(-np.inf)
            ub.append(0.0)

    # ------------------------------------------------------------
    # BINARY INSTALLATION LINKING: Pcap/Ecap = 0 when technology is off
    # ------------------------------------------------------------
    for k in range(K):
        A.append({Pcap_idx[k]: 1.0, install_idx[k]: -MAX_STORAGE_POWER_MW})
        lb.append(-np.inf)
        ub.append(0.0)
        A.append({Ecap_idx[k]: 1.0, install_idx[k]: -MAX_STORAGE_ENERGY_MWH})
        lb.append(-np.inf)
        ub.append(0.0)

    # ------------------------------------------------------------
    # DURATION CONSTRAINTS
    # ------------------------------------------------------------

    for k, tech in enumerate(technologies):

        p = storage_parameters[tech]

        min_duration = p["min_duration"]
        max_duration = p["max_duration"]

        # E >= minimum duration × P
        row = {
            Pcap_idx[k]: min_duration,
            Ecap_idx[k]: -1.0
        }

        A.append(row)
        lb.append(-np.inf)
        ub.append(0.0)

        # E <= maximum duration × P
        row = {
            Ecap_idx[k]: 1.0,
            Pcap_idx[k]: -max_duration
        }

        A.append(row)
        lb.append(-np.inf)
        ub.append(0.0)

    # ------------------------------------------------------------
    # OPTIONAL FIXED-HYBRID ENERGY-SHARE CONSTRAINTS
    # ------------------------------------------------------------
    # fixed_energy_shares maps technology names to shares summing to 1.
    # Enforces E_k = share_k * total selected energy capacity.
    if fixed_energy_shares is not None:
        if set(fixed_energy_shares) != set(technologies):
            raise ValueError("Fixed-hybrid shares must specify every selected technology.")
        share_sum = sum(float(v) for v in fixed_energy_shares.values())
        if not np.isclose(share_sum, 1.0, atol=1e-8):
            raise ValueError("Fixed-hybrid energy shares must sum to 1.")
        for k, tech in enumerate(technologies):
            row = {Ecap_idx[k]: 1.0}
            for j, other in enumerate(technologies):
                row[Ecap_idx[j]] = row.get(Ecap_idx[j], 0.0) - float(fixed_energy_shares[tech])
            A.append(row)
            lb.append(0.0)
            ub.append(0.0)

    # ------------------------------------------------------------
    # CONVERT SPARSE DICTIONARY TO MATRIX
    # ------------------------------------------------------------

    from scipy.sparse import lil_matrix

    n_constraints = len(A)

    A_sparse = lil_matrix(
        (n_constraints, nvars)
    )

    for i, row in enumerate(A):

        for j, value in row.items():

            A_sparse[i, j] = value

    A_sparse = A_sparse.tocsr()

    constraints = LinearConstraint(
        A_sparse,
        np.array(lb),
        np.array(ub)
    )

    # ------------------------------------------------------------
    # SOLVE
    # ------------------------------------------------------------

    print("Running mixed-integer/linear optimisation...")

    integrality = np.zeros(nvars, dtype=int)
    integrality[install_idx] = 1
    result = milp(
        c=c,
        integrality=integrality,
        bounds=Bounds(
            lower,
            upper
        ),
        constraints=constraints,
        options={
            "time_limit": 300,
            "mip_rel_gap": 0.01
        }
    )

    if not result.success:

        print("\nOptimisation status:")
        print(result.message)

        raise RuntimeError(
            "DASPI optimisation did not converge: " + str(result.message)
        )

    print("\nDASPI optimisation completed.")
    print(result.message)

    x = result.x

    # ------------------------------------------------------------
    # EXTRACT RESULTS
    # ------------------------------------------------------------

    portfolio = []

    for k, tech in enumerate(technologies):

        P = x[Pcap_idx[k]]
        E = x[Ecap_idx[k]]

        duration = (
            E / P
            if P > 1e-6
            else 0
        )

        portfolio.append({

            "Technology": tech,

            "Power_MW": P,

            "Energy_MWh": E,

            "Duration_h": duration,

            "Annualised_Capex_USD": (
                P
                * storage_parameters[tech]["power_capex"]
                * 1000
                / MODEL_LIFETIME_YEARS
                +
                E
                * storage_parameters[tech]["energy_capex"]
                * 1000
                / MODEL_LIFETIME_YEARS
            )
        })

    portfolio_df = pd.DataFrame(
        portfolio
    )

    dispatch = data.copy()

    dispatch["renewable_used_MW"] = (
        x[renewable_used_idx]
    )

    dispatch["grid_MW"] = (
        x[grid_idx]
    )

    dispatch["unmet_MW"] = (
        x[unmet_idx]
    )

    for k, tech in enumerate(technologies):

        dispatch[
            f"{tech}_charge_MW"
        ] = x[
            charge_idx[k, :]
        ]

        dispatch[
            f"{tech}_discharge_MW"
        ] = x[
            discharge_idx[k, :]
        ]

        dispatch[
            f"{tech}_SOC_MWh"
        ] = x[
            soc_idx[k, :]
        ]

    dispatch["curtailment_MW"] = np.maximum(
        dispatch["renewable_available_MW"]
        -
        dispatch["renewable_used_MW"],
        0
    )

    return result, portfolio_df, dispatch


# ================================================================
# 15. RUN DASPI
# ================================================================

result, portfolio_df, dispatch = build_and_solve_DASPI(
    df,
    TECHNOLOGIES,
    STORAGE
)


# ================================================================
# 16. PORTFOLIO RESULTS
# ================================================================

print("\n================================================")
print("DASPI OPTIMAL STORAGE PORTFOLIO")
print("================================================")

print(
    portfolio_df.to_string(
        index=False,
        formatters={
            "Power_MW": "{:.2f}".format,
            "Energy_MWh": "{:.2f}".format,
            "Duration_h": "{:.2f}".format,
            "Annualised_Capex_USD":
                "{:,.0f}".format
        }
    )
)


# ================================================================
# 17. SYSTEM PERFORMANCE METRICS
# ================================================================

annual_curtailment = (
    dispatch["curtailment_MW"].sum()
    * RESAMPLE_HOURS
)

annual_grid_energy = (
    dispatch["grid_MW"].sum()
    * RESAMPLE_HOURS
)

annual_unmet_energy = (
    dispatch["unmet_MW"].sum()
    * RESAMPLE_HOURS
)

annual_load = (
    dispatch["demand_MW"].sum()
    * RESAMPLE_HOURS
)

annual_renewable_available = (
    dispatch["renewable_available_MW"].sum()
    * RESAMPLE_HOURS
)

annual_renewable_used = (
    dispatch["renewable_used_MW"].sum()
    * RESAMPLE_HOURS
)

renewable_utilisation = (
    annual_renewable_used
    /
    annual_renewable_available
    if annual_renewable_available > 0
    else 0
)

reliability = (
    1
    -
    annual_unmet_energy
    /
    annual_load
)

curtailment_rate = (
    annual_curtailment
    /
    annual_renewable_available
)

total_storage_energy = (
    portfolio_df["Energy_MWh"].sum()
)

total_storage_power = (
    portfolio_df["Power_MW"].sum()
)


# ================================================================
# 18. TSCR — EVENT-COVERAGE SCREEN, NOT ENERGY-CAPACITY RATIO
# ================================================================
# TSCR is defined here as the fraction of detected deficit episodes that
# pass a transparent optimistic screening test: peak deficit <= aggregate
# storage power and episode deficit energy <= estimated usable storage energy.
# Each episode is treated independently and assumed to begin fully charged;
# therefore this is NOT a dispatch-validated reliability metric.

def calculate_TSCR_event_screen(events_df, portfolio_df):
    if events_df is None or len(events_df) == 0:
        return np.nan
    if portfolio_df is None or len(portfolio_df) == 0:
        return 0.0
    p_total = float(portfolio_df["Power_MW"].sum())
    usable_e = 0.0
    for _, row in portfolio_df.iterrows():
        tech = row["Technology"]
        usable_e += float(row["Energy_MWh"]) * float(STORAGE[tech]["eta_dis"])
    covered = (
        (events_df["peak_deficit_MW"].to_numpy() <= p_total + 1e-9)
        & (events_df["energy_deficit_MWh"].to_numpy() <= usable_e + 1e-9)
    )
    return float(np.mean(covered))

TSCR = calculate_TSCR_event_screen(events, portfolio_df)
# Small synthetic test verifies the TSCR event-screen definition.
_tscr_test_events = pd.DataFrame({
    "peak_deficit_MW": [10.0, 100.0],
    "energy_deficit_MWh": [50.0, 500.0]
})
_tscr_test_portfolio = pd.DataFrame({
    "Technology": ["Li-ion"], "Power_MW": [20.0], "Energy_MWh": [100.0]
})
_tscr_test_value = calculate_TSCR_event_screen(_tscr_test_events, _tscr_test_portfolio)
assert np.isclose(_tscr_test_value, 0.5), f"TSCR unit test failed: {_tscr_test_value}"
print("TSCR event-screen unit test passed (synthetic test only):", _tscr_test_value)
TSCR_DEFINITION = (
    "fraction of detected deficit episodes passing peak-power and usable-energy "
    "screen; each event assumed independently fully charged; not dispatch validated"
)

# ================================================================
# 19. RASV — COMPUTED ONLY AFTER NO-STORAGE BASELINE
# ================================================================
# RASV = avoided unserved energy (MWh) / annualised storage CAPEX (million USD).
# Units: MWh per million USD over the modeled horizon. This is a transparent
# cost-effectiveness indicator, not a dimensionless universal resilience score.
# It must be computed from matched, independently solved no-storage and case
# dispatches. Until that baseline run exists, RASV is deliberately undefined.
annual_storage_cost = float(portfolio_df["Annualised_Capex_USD"].sum())
RASV = np.nan
RASV_DEFINITION = (
    "avoided unserved energy relative to independently solved No Storage case "
    "(MWh) divided by annualised storage CAPEX (million USD); report horizon and inputs"
)

# ================================================================
# 20. PRINT FINAL DASPI METRICS
# ================================================================

print("\n================================================")
print("DASPI PERFORMANCE INDICATORS")
print("================================================")

print(
    f"DMI  = {DMI:.4f}"
)

print(
    f"RASV = {RASV:.6f}"
)

print(
    f"TSCR = {TSCR:.4f}"
)

print(
    f"Reliability = {reliability*100:.4f}%"
)

print(
    f"Renewable utilisation = "
    f"{renewable_utilisation*100:.2f}%"
)

print(
    f"Curtailment = "
    f"{annual_curtailment:,.0f} MWh"
)

print(
    f"Grid import = "
    f"{annual_grid_energy:,.0f} MWh"
)

print(
    f"Unserved energy = "
    f"{annual_unmet_energy:,.6f} MWh"
)


# ================================================================
# 21. TECHNOLOGY CONTRIBUTION
# ================================================================

portfolio_df["Share_of_Energy_Capacity_%"] = (
    portfolio_df["Energy_MWh"]
    /
    max(
        portfolio_df["Energy_MWh"].sum(),
        1e-9
    )
    * 100
)

portfolio_df["Share_of_Power_Capacity_%"] = (
    portfolio_df["Power_MW"]
    /
    max(
        portfolio_df["Power_MW"].sum(),
        1e-9
    )
    * 100
)

print("\n================================================")
print("STORAGE PORTFOLIO SHARES")
print("================================================")

print(
    portfolio_df[
        [
            "Technology",
            "Power_MW",
            "Energy_MWh",
            "Duration_h",
            "Share_of_Energy_Capacity_%",
            "Share_of_Power_Capacity_%"
        ]
    ].to_string(
        index=False,
        formatters={
            "Power_MW": "{:.2f}".format,
            "Energy_MWh": "{:.2f}".format,
            "Duration_h": "{:.2f}".format,
            "Share_of_Energy_Capacity_%":
                "{:.2f}".format,
            "Share_of_Power_Capacity_%":
                "{:.2f}".format
        }
    )
)


# ================================================================
# 22. COMPARATIVE SCENARIO MODEL
# ================================================================
#
# This section creates transparent comparison cases.
#
# The DASPI case comes from the optimisation.
#
# Other cases:
# 1. No storage
# 2. Li-ion only
# 3. Flow only
# 4. Pumped hydro only
# 5. Hydrogen only
# 6. Thermal only
# 7. DASPI
#
# We evaluate storage portfolio capacity against the observed
# deficit signature.
#
# NOTE:
# These are diagnostic comparisons, not substitutes for a
# full optimisation of every scenario.

def evaluate_storage_scenario(
    portfolio,
    events,
    deficit_series,
    timestep_hours
):

    if len(portfolio) == 0:

        energy_capacity = 0
        power_capacity = 0

    else:

        energy_capacity = (
            portfolio["Energy_MWh"].sum()
        )

        power_capacity = (
            portfolio["Power_MW"].sum()
        )

    total_deficit = (
        deficit_series.sum()
        * timestep_hours
    )

    if total_deficit <= 0:

        coverage = 1.0

    else:

        coverage = min(
            energy_capacity
            /
            total_deficit,
            1.0
        )

    # Approximate duration coverage
    if power_capacity > 0:

        effective_duration = (
            energy_capacity
            /
            power_capacity
        )

    else:

        effective_duration = 0

    # Event coverage
    if len(events) > 0:

        event_coverage = np.mean(
            events["duration_h"]
            <=
            max(
                effective_duration,
                0
            )
        )

    else:

        event_coverage = 1.0

    return {
        "Energy_Capacity_MWh":
            energy_capacity,

        "Power_Capacity_MW":
            power_capacity,

        "Energy_Coverage":
            coverage,

        "Effective_Duration_h":
            effective_duration,

        "Event_Coverage":
            event_coverage
    }


scenarios = []

# No storage
no_storage = pd.DataFrame(
    columns=portfolio_df.columns
)

metrics = evaluate_storage_scenario(
    no_storage,
    events,
    df["deficit_MW"],
    RESAMPLE_HOURS
)

scenarios.append({
    "Scenario": "No Storage",
    **metrics
})


# Single technology scenarios
for tech in TECHNOLOGIES:

    single = portfolio_df[
        portfolio_df["Technology"] == tech
    ].copy()

    metrics = evaluate_storage_scenario(
        single,
        events,
        df["deficit_MW"],
        RESAMPLE_HOURS
    )

    scenarios.append({
        "Scenario": tech,
        **metrics
    })


# DASPI
metrics = evaluate_storage_scenario(
    portfolio_df,
    events,
    df["deficit_MW"],
    RESAMPLE_HOURS
)

scenarios.append({
    "Scenario": "DASPI Optimised Portfolio",
    **metrics
})


scenario_df = pd.DataFrame(
    scenarios
)

print("\n================================================")
print("NON-OPTIMISED CAPACITY SCREEN — NOT INDEPENDENT BASELINES")
print("================================================")
print("These rows are descriptive screens only. Use the independent MILP baseline harness for comparative claims.")

print(
    scenario_df.to_string(
        index=False,
        formatters={
            "Energy_Capacity_MWh":
                "{:,.2f}".format,
            "Power_Capacity_MW":
                "{:,.2f}".format,
            "Energy_Coverage":
                "{:.3f}".format,
            "Effective_Duration_h":
                "{:.2f}".format,
            "Event_Coverage":
                "{:.3f}".format
        }
    )
)


# ================================================================
# 23. SENSITIVITY ANALYSIS
# ================================================================
#
# Sensitivity to renewable penetration.
#
# The optimisation is rerun using different renewable targets.
#
# This is computationally expensive.
#
# For the first Colab run, use 50/70/90%.
#
# For publication, run 30/40/50/60/70/80/90%.

RE_TARGETS = [
    0.50,
    0.70,
    0.90
]

sensitivity_results = []

for target in RE_TARGETS:

    print(
        f"\nRunning sensitivity case: "
        f"{target*100:.0f}% renewable penetration"
    )

    annual_load = (
        df["demand_MW"].sum()
        * RESAMPLE_HOURS
    )

    annual_hydro = (
        df["hydro_MW"].sum()
        * RESAMPLE_HOURS
    )

    target_RE = (
        target
        *
        annual_load
    )

    remaining = max(
        target_RE
        -
        annual_hydro,
        0
    )

    solar_target = (
        remaining
        *
        SOLAR_SHARE
    )

    wind_target = (
        remaining
        *
        WIND_SHARE
    )

    solar_cap = (
        solar_target
        /
        (
            df["solar_cf"].mean()
            *
            8760
        )
    )

    wind_cap = (
        wind_target
        /
        (
            df["wind_cf"].mean()
            *
            8760
        )
    )

    solar_cap = min(
        solar_cap,
        MAX_SOLAR_MW
    )

    wind_cap = min(
        wind_cap,
        MAX_WIND_MW
    )

    test = df.copy()

    test["solar_MW"] = (
        test["solar_cf"]
        *
        solar_cap
    )

    test["wind_MW"] = (
        test["wind_cf"]
        *
        wind_cap
    )

    test["renewable_available_MW"] = (
        test["solar_MW"]
        +
        test["wind_MW"]
        +
        test["hydro_MW"]
    )

    test["deficit_MW"] = np.maximum(
        test["demand_MW"]
        -
        test["renewable_available_MW"],
        0
    )

    test["surplus_MW"] = np.maximum(
        test["renewable_available_MW"]
        -
        test["demand_MW"],
        0
    )

    test_events = detect_deficit_events(
        test["deficit_MW"],
        RESAMPLE_HOURS,
        0
    )

    test_DMI = calculate_DMI(
        test_events
    )

    sensitivity_results.append({
        "Renewable_Target":
            target,

        "Solar_MW":
            solar_cap,

        "Wind_MW":
            wind_cap,

        "DMI":
            test_DMI,

        "Number_Deficit_Events":
            len(test_events),

        "Longest_Deficit_h":
            (
                test_events["duration_h"].max()
                if len(test_events) > 0
                else 0
            )
    })


sensitivity_df = pd.DataFrame(
    sensitivity_results
)

print("\n================================================")
print("RENEWABLE PENETRATION SENSITIVITY")
print("================================================")

print(
    sensitivity_df.to_string(
        index=False,
        formatters={
            "Renewable_Target":
                "{:.0%}".format,
            "Solar_MW":
                "{:.2f}".format,
            "Wind_MW":
                "{:.2f}".format,
            "DMI":
                "{:.4f}".format,
            "Longest_Deficit_h":
                "{:.2f}".format
        }
    )
)


# ================================================================
# 24. FIGURE 1 — RENEWABLE + LOAD PROFILE
# ================================================================

plt.figure(figsize=(15, 5))

sample = df.iloc[:7*24//RESAMPLE_HOURS]

plt.plot(
    sample.index,
    sample["demand_MW"],
    label="Demand"
)

plt.plot(
    sample.index,
    sample["solar_MW"],
    label="Solar"
)

plt.plot(
    sample.index,
    sample["wind_MW"],
    label="Wind"
)

plt.plot(
    sample.index,
    sample["hydro_MW"],
    label="Hydro"
)

plt.plot(
    sample.index,
    sample["renewable_available_MW"],
    label="Total Renewable"
)

plt.title(
    "DASPI: Renewable Generation and Electricity Demand"
)

plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "Figure_1_Renewable_Load_Profile.png",
    dpi=300
)

plt.show()


# ================================================================
# 25. FIGURE 2 — RENEWABLE DEFICIT SIGNATURE
# ================================================================

plt.figure(figsize=(15, 5))

sample = df.iloc[:14*24//RESAMPLE_HOURS]

plt.fill_between(
    sample.index,
    sample["deficit_MW"],
    0,
    alpha=0.35,
    label="Renewable Deficit"
)

plt.plot(
    sample.index,
    sample["demand_MW"],
    label="Demand"
)

plt.plot(
    sample.index,
    sample["renewable_available_MW"],
    label="Renewable Supply"
)

plt.title(
    "DASPI Deficit Signature: Temporal Renewable-Demand Mismatch"
)

plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "Figure_2_Deficit_Signature.png",
    dpi=300
)

plt.show()


# ================================================================
# 26. FIGURE 3 — DEFICIT DURATION DISTRIBUTION
# ================================================================

if len(events) > 0:

    plt.figure(figsize=(10, 5))

    plt.hist(
        events["duration_h"],
        bins=15
    )

    plt.axvline(
        REFERENCE_DURATION_HOURS,
        linestyle="--",
        label="8-hour reference"
    )

    plt.title(
        "Distribution of Renewable-Deficit Event Durations"
    )

    plt.xlabel(
        "Deficit Duration (hours)"
    )

    plt.ylabel(
        "Number of Events"
    )

    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        "Figure_3_Deficit_Duration_Distribution.png",
        dpi=300
    )

    plt.show()


# ================================================================
# 27. FIGURE 4 — OPTIMISED STORAGE PORTFOLIO
# ================================================================

plt.figure(figsize=(11, 6))

plt.bar(
    portfolio_df["Technology"],
    portfolio_df["Energy_MWh"]
)

plt.title(
    "DASPI Optimised Storage Energy Capacity"
)

plt.xlabel(
    "Storage Technology"
)

plt.ylabel(
    "Energy Capacity (MWh)"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "Figure_4_Optimised_Storage_Portfolio.png",
    dpi=300
)

plt.show()


# ================================================================
# 28. FIGURE 5 — STORAGE DURATION
# ================================================================

plt.figure(figsize=(11, 6))

plt.bar(
    portfolio_df["Technology"],
    portfolio_df["Duration_h"]
)

plt.title(
    "Emergent Storage Duration in DASPI Portfolio"
)

plt.xlabel(
    "Storage Technology"
)

plt.ylabel(
    "Optimised Duration (hours)"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "Figure_5_Storage_Duration.png",
    dpi=300
)

plt.show()


# ================================================================
# 29. FIGURE 6 — CHARGING / DISCHARGING
# ================================================================

sample = dispatch.iloc[
    :14*24//RESAMPLE_HOURS
]

plt.figure(figsize=(15, 6))

for tech in TECHNOLOGIES:

    plt.plot(
        sample.index,
        sample[
            f"{tech}_discharge_MW"
        ],
        label=f"{tech} discharge"
    )

plt.plot(
    sample.index,
    sample["demand_MW"],
    linestyle="--",
    label="Demand"
)

plt.title(
    "DASPI Multi-Technology Storage Dispatch"
)

plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend(
    ncol=2,
    fontsize=8
)

plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "Figure_6_Storage_Dispatch.png",
    dpi=300
)

plt.show()


# ================================================================
# 30. FIGURE 7 — DMI VS RENEWABLE PENETRATION
# ================================================================

plt.figure(figsize=(9, 5))

plt.plot(
    sensitivity_df["Renewable_Target"] * 100,
    sensitivity_df["DMI"],
    marker="o"
)

plt.title(
    "Duration Mismatch Index vs Renewable Penetration"
)

plt.xlabel(
    "Renewable Penetration Target (%)"
)

plt.ylabel(
    "Duration Mismatch Index (DMI)"
)

plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "Figure_7_DMI_Sensitivity.png",
    dpi=300
)

plt.show()


# ================================================================
# 31. FIGURE 8 — COORDINATE RESOURCE DIAGNOSTIC
# ================================================================

monthly_resource = (
    resource
    .resample("ME")
    .agg({
        "solar_resource": "mean",
        "wind_speed": "mean"
    })
)

fig, ax1 = plt.subplots(
    figsize=(12, 5)
)

ax1.plot(
    monthly_resource.index,
    monthly_resource["solar_resource"],
    marker="o",
    label="Solar resource"
)

ax1.set_xlabel(
    "Month"
)

ax1.set_ylabel(
    "Solar resource indicator"
)

ax2 = ax1.twinx()

ax2.plot(
    monthly_resource.index,
    monthly_resource["wind_speed"],
    marker="s",
    linestyle="--",
    label="Wind speed"
)

ax2.set_ylabel(
    "Wind speed (m/s)"
)

plt.title(
    f"Coordinate-Specific Renewable Resource Profile: "
    f"{LOCATION_NAME}"
)

fig.tight_layout()

plt.savefig(
    "Figure_8_Coordinate_Resource_Profile.png",
    dpi=300
)

plt.show()


# ================================================================
# 32. SAVE RESULTS
# ================================================================

df.to_csv(
    "DASPI_system_timeseries.csv"
)

events.to_csv(
    "DASPI_deficit_events.csv",
    index=False
)

portfolio_df.to_csv(
    "DASPI_optimal_storage_portfolio.csv",
    index=False
)

scenario_df.to_csv(
    "DASPI_scenario_comparison.csv",
    index=False
)

sensitivity_df.to_csv(
    "DASPI_sensitivity_results.csv",
    index=False
)


# ================================================================
# 33. FINAL RESEARCH SUMMARY
# ================================================================

summary = {

    "Location":
        LOCATION_NAME,

    "Latitude":
        LATITUDE,

    "Longitude":
        LONGITUDE,

    "Renewable_Target":
        TARGET_RENEWABLE_SHARE,

    "Solar_Capacity_MW":
        solar_capacity,

    "Wind_Capacity_MW":
        wind_capacity,

    "Hydro_Capacity_MW":
        HYDRO_CAPACITY_MW,

    "DMI":
        DMI,

    "RASV":
        RASV,

    "TSCR":
        TSCR,

    "Reliability":
        reliability,

    "Renewable_Utilisation":
        renewable_utilisation,

    "Curtailment_MWh":
        annual_curtailment,

    "Grid_Import_MWh":
        annual_grid_energy,

    "Unserved_Energy_MWh":
        annual_unmet_energy,

    "Total_Storage_Power_MW":
        total_storage_power,

    "Total_Storage_Energy_MWh":
        total_storage_energy
}

summary_df = pd.DataFrame(
    [summary]
)

summary_df.to_csv(
    "DASPI_research_summary.csv",
    index=False
)

print("\n")
print("=" * 70)
print("DASPI RESEARCH RUN COMPLETED")
print("=" * 70)

for key, value in summary.items():

    if isinstance(value, float):

        print(
            f"{key:35s}: {value:.4f}"
        )

    else:

        print(
            f"{key:35s}: {value}"
        )

print("\nGenerated files:")

files = [
    "DASPI_system_timeseries.csv",
    "DASPI_deficit_events.csv",
    "DASPI_optimal_storage_portfolio.csv",
    "DASPI_scenario_comparison.csv",
    "DASPI_sensitivity_results.csv",
    "DASPI_research_summary.csv",
    "Figure_1_Renewable_Load_Profile.png",
    "Figure_2_Deficit_Signature.png",
    "Figure_3_Deficit_Duration_Distribution.png",
    "Figure_4_Optimised_Storage_Portfolio.png",
    "Figure_5_Storage_Duration.png",
    "Figure_6_Storage_Dispatch.png",
    "Figure_7_DMI_Sensitivity.png",
    "Figure_8_Coordinate_Resource_Profile.png"
]

for f in files:
    print("  ✓", f)

print("\nDASPI pipeline:")
print(
    "Coordinates → Resource Data → Renewable Profile → "
    "Deficit Signature → DMI → MILP Optimisation → "
    "Storage Portfolio → RASV/TSCR → Sensitivity"
)

print("\nIMPORTANT FOR PUBLICATION:")
print(
    "Replace the illustrative demand, hydro and technology-cost "
    "assumptions with cited datasets/parameters before reporting "
    "the numerical results as scientific findings."
)

# ============================================================

# ============================================================
# INDEPENDENTLY SOLVED COMPARATIVE BASELINES (REVIEWER REVISION)
# ============================================================
# All cases use the same `df`, demand/renewable profiles, grid policy,
# objective coefficients, timestep and solver formulation as DASPI-C.
# No-storage may still use grid imports; report grid-supported results explicitly.
# Fixed Hybrid uses a predeclared 50% Li-ion / 10% Flow / 40% Thermal
# split of selected energy capacity. This is a modelling assumption, not an
# empirically validated technology allocation.

import traceback

BASELINE_DIR = "/content/DASPI_Baseline_Runs"
os.makedirs(BASELINE_DIR, exist_ok=True)

baseline_specs = [
    ("No Storage", [], {}),
    ("Li-ion Only", ["Li-ion"], {}),
    ("Single LDES - Hydrogen", ["Hydrogen"], {}),
    ("Fixed Hybrid", ["Li-ion", "Flow Battery", "Thermal Storage"],
     {"Li-ion": 0.50, "Flow Battery": 0.10, "Thermal Storage": 0.40}),
]

baseline_summary_rows = []
baseline_portfolio_frames = []
baseline_dispatch_frames = []
baseline_failures = []

for baseline_name, baseline_techs, fixed_shares in baseline_specs:
    print(f"\n--- Solving baseline: {baseline_name} ---")
    try:
        run_result, run_portfolio, run_dispatch = build_and_solve_DASPI(
            df.copy(),
            baseline_techs,
            STORAGE,
            fixed_energy_shares=(fixed_shares if fixed_shares else None)
        )

        # Energy quantities are integrated over the model timestep.
        dt_hours = RESAMPLE_HOURS
        unserved_mwh = float(run_dispatch["unmet_MW"].sum() * dt_hours)
        grid_import_mwh = float(run_dispatch["grid_MW"].sum() * dt_hours)
        curtailment_mwh = float(run_dispatch["curtailment_MW"].sum() * dt_hours)
        demand_mwh = float(run_dispatch["demand_MW"].sum() * dt_hours)
        reliability_pct = (100.0 * (1.0 - unserved_mwh / demand_mwh)
                           if demand_mwh > 0 else float("nan"))
        annualised_storage_cost = float(run_portfolio["Annualised_Capex_USD"].sum()) if len(run_portfolio) else 0.0
        annualised_grid_cost = float(grid_import_mwh * 0.15)
        total_modelled_cost = annualised_storage_cost + annualised_grid_cost

        baseline_summary_rows.append({
            "Baseline": baseline_name,
            "Solver_Status": str(run_result.message),
            "Solver_Success": bool(run_result.success),
            "Annualised_Storage_CAPEX_USD": annualised_storage_cost,
            "Grid_Import_Cost_USD_proxy": annualised_grid_cost,
            "Total_Cost_USD_proxy": total_modelled_cost,
            "Curtailment_MWh": curtailment_mwh,
            "Unserved_Energy_MWh": unserved_mwh,
            "Reliability_pct_grid_supported": reliability_pct,
            "Grid_Imports_MWh": grid_import_mwh,
            "Storage_Power_MW": float(run_portfolio["Power_MW"].sum()) if len(run_portfolio) else 0.0,
            "Storage_Energy_MWh": float(run_portfolio["Energy_MWh"].sum()) if len(run_portfolio) else 0.0,
            "Fixed_Energy_Shares": json.dumps(fixed_shares) if fixed_shares else ""
        })
        if len(run_portfolio):
            tmp = run_portfolio.copy()
            tmp.insert(0, "Baseline", baseline_name)
            baseline_portfolio_frames.append(tmp)
        tmp_dispatch = run_dispatch.copy()
        tmp_dispatch.insert(0, "Baseline", baseline_name)
        baseline_dispatch_frames.append(tmp_dispatch)

    except Exception as exc:
        baseline_failures.append({
            "Baseline": baseline_name,
            "Error": repr(exc),
            "Traceback": traceback.format_exc()
        })
        print(f"BASELINE RUN FAILED: {baseline_name}: {exc}")

baseline_summary = pd.DataFrame(baseline_summary_rows)
baseline_summary.to_csv(os.path.join(BASELINE_DIR, "independently_solved_baseline_summary.csv"), index=False)

if baseline_portfolio_frames:
    pd.concat(baseline_portfolio_frames, ignore_index=True).to_csv(
        os.path.join(BASELINE_DIR, "independently_solved_baseline_portfolios.csv"), index=False
    )
if baseline_dispatch_frames:
    pd.concat(baseline_dispatch_frames, ignore_index=True).to_csv(
        os.path.join(BASELINE_DIR, "independently_solved_baseline_dispatch.csv"), index=False
    )
if baseline_failures:
    pd.DataFrame(baseline_failures).to_json(
        os.path.join(BASELINE_DIR, "baseline_solver_failures.json"), orient="records", indent=2
    )

print("\nBASELINE RUN SUMMARY")
print(baseline_summary.to_string(index=False) if len(baseline_summary) else "No baseline completed successfully.")
print("Saved baseline artefacts to:", BASELINE_DIR)


# DASPI / SPATIAL-TEMPORAL STORAGE OPTIMIZATION
# OUTPUT SAVING MODULE
# ============================================================

import os
import json
import shutil
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CREATE OUTPUT DIRECTORIES
# ------------------------------------------------------------

BASE_DIR = "/content/DASPI_Research_Outputs"

folders = {
    "tables": os.path.join(BASE_DIR, "tables"),
    "figures": os.path.join(BASE_DIR, "figures"),
    "metrics": os.path.join(BASE_DIR, "metrics"),
    "models": os.path.join(BASE_DIR, "models"),
    "reports": os.path.join(BASE_DIR, "reports")
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

print("Output directories created:")
for name, path in folders.items():
    print(f"  {name}: {path}")


# ============================================================
# 2. HELPER FUNCTION FOR SAFE SAVING
# ============================================================

def save_dataframe(df, filename):
    """
    Save a DataFrame as CSV and Excel.
    """
    if df is None:
        print(f"Skipping {filename}: DataFrame is None.")
        return

    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    csv_path = os.path.join(folders["tables"], filename + ".csv")
    xlsx_path = os.path.join(folders["tables"], filename + ".xlsx")

    df.to_csv(csv_path, index=False)

    try:
        df.to_excel(xlsx_path, index=False)
    except Exception as e:
        print("Excel export skipped:", e)

    print(f"Saved: {filename}")


# ============================================================
# 3. SAVE EXISTING DATASETS / RESULTS
# ============================================================

# The following section automatically detects common variable
# names used in the DASPI notebook.

candidate_dataframes = {
    "renewable_data": globals().get("renewable_data"),
    "load_data": globals().get("load_data"),
    "deficit_data": globals().get("deficit_data"),
    "dispatch_results": globals().get("dispatch_results"),
    "portfolio_results": globals().get("portfolio_results"),
    "optimization_results": globals().get("optimization_results"),
    "scenario_results": globals().get("scenario_results"),
    "results_df": globals().get("results_df"),
    "metrics_df": globals().get("metrics_df"),
    "storage_results": globals().get("storage_results"),
    "coordinate_results": globals().get("coordinate_results"),
}

for name, df in candidate_dataframes.items():
    if df is not None:
        save_dataframe(df, name)


# ============================================================
# 4. SAVE STORAGE PORTFOLIO
# ============================================================

storage_portfolio = globals().get("storage_portfolio")

if storage_portfolio is not None:

    if isinstance(storage_portfolio, dict):
        portfolio_df = pd.DataFrame(
            list(storage_portfolio.items()),
            columns=["Technology", "Capacity"]
        )
    else:
        portfolio_df = pd.DataFrame(storage_portfolio)

    save_dataframe(
        portfolio_df,
        "DASPI_Optimal_Storage_Portfolio"
    )


# ============================================================
# 5. SAVE COORDINATE / SPATIAL RESULTS
# ============================================================

coordinates = globals().get("coordinates")

if coordinates is not None:

    try:

        if isinstance(coordinates, dict):

            coordinate_df = pd.DataFrame(
                [
                    {
                        "Location": k,
                        **(
                            v if isinstance(v, dict)
                            else {"Value": v}
                        )
                    }
                    for k, v in coordinates.items()
                ]
            )

        else:
            coordinate_df = pd.DataFrame(coordinates)

        save_dataframe(
            coordinate_df,
            "Spatial_Coordinate_Results"
        )

    except Exception as e:
        print("Coordinate results could not be exported:", e)


# ============================================================
# 6. SAVE OPTIMIZATION METRICS
# ============================================================

metrics = {}

# Try to capture commonly used metrics
possible_metrics = [
    "total_cost",
    "system_cost",
    "annual_cost",
    "renewable_curtailment",
    "curtailment",
    "co2_emissions",
    "emissions",
    "renewable_utilization",
    "renewable_utilisation",
    "reliability",
    "unmet_demand",
    "storage_capacity",
    "storage_power",
    "DMI",
    "TSCR",
    "RASV"
]

for variable in possible_metrics:

    if variable in globals():

        value = globals()[variable]

        try:
            if np.isscalar(value):
                metrics[variable] = float(value)
            else:
                metrics[variable] = value
        except:
            metrics[variable] = str(value)


# Save JSON
metrics_path = os.path.join(
    folders["metrics"],
    "DASPI_Key_Metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4, default=str)

print("\nSaved metrics:")
print(metrics_path)


# ============================================================
# 7. SAVE DMI / TSCR / RASV
# ============================================================

diagnostic_metrics = {}

for metric_name in ["DMI", "TSCR", "RASV"]:

    if metric_name in globals():

        diagnostic_metrics[metric_name] = globals()[metric_name]

diagnostic_df = pd.DataFrame(
    list(diagnostic_metrics.items()),
    columns=["Metric", "Value"]
)

if len(diagnostic_df) > 0:

    save_dataframe(
        diagnostic_df,
        "DASPI_Diagnostic_Metrics"
    )


# ============================================================
# 8. SAVE ALL CURRENT FIGURES
# ============================================================

print("\nSearching for generated figures...")

figure_counter = 0

for num in plt.get_fignums():

    fig = plt.figure(num)

    figure_counter += 1

    filename = os.path.join(
        folders["figures"],
        f"DASPI_Figure_{figure_counter:02d}.png"
    )

    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    print("Saved:", filename)


print(f"\nTotal figures saved: {figure_counter}")


# ============================================================
# 9. CREATE RESEARCH SUMMARY
# ============================================================

summary = {
    "Project":
        "Spatial-Temporal Storage Optimization for Renewable Power Integration",

    "Framework":
        "Duration-Adaptive Storage Portfolio Intelligence (DASPI)",

    "Research_Extension":
        "Spatial-Temporal Coordinate-Constrained Optimization",

    "Storage_Technologies": [
        "Lithium-ion BESS",
        "Vanadium Redox Flow Battery",
        "Pumped Hydro Storage",
        "Hydrogen Storage",
        "Thermal Energy Storage"
    ],

    "Diagnostic_Metrics": [
        "Duration Mismatch Index (DMI)",
        "Temporal Storage Coverage Ratio (TSCR)",
        "Resilience-Adjusted Storage Value (RASV)"
    ],

    "Optimization_Dimensions": [
        "Cost",
        "Reliability",
        "Renewable Curtailment",
        "CO2 Emissions",
        "Storage Capacity",
        "Storage Duration",
        "Spatial Coordinates"
    ]
}

summary_path = os.path.join(
    folders["reports"],
    "DASPI_Research_Summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Research summary saved.")


# ============================================================
# 10. CREATE README FOR OUTPUT FOLDER
# ============================================================

readme_text = """
# DASPI Spatial-Temporal Storage Optimization

Project:
Spatial-Temporal Storage Optimization for Renewable Power Integration

Framework:
Duration-Adaptive Storage Portfolio Intelligence (DASPI)

The output package contains:

## tables/
CSV and Excel research datasets and optimization results.

## figures/
Publication-quality PNG figures generated by the model.

## metrics/
DMI, TSCR, RASV and other optimization metrics.

## models/
Reserved for trained ML models and serialized optimization models.

## reports/
Research summary and supporting outputs.

Main research dimensions:

- Renewable penetration
- Storage duration
- Multi-technology storage portfolio
- Renewable deficit events
- Spatial coordinates
- Storage siting
- Cost
- Reliability
- Renewable curtailment
- CO2 emissions
- Storage degradation
"""

readme_path = os.path.join(
    BASE_DIR,
    "README.txt"
)

with open(readme_path, "w") as f:
    f.write(readme_text)

print("README created.")


# ============================================================
# 11. ZIP EVERYTHING
# ============================================================

zip_path = "/content/DASPI_Spatial_Temporal_Research_Outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(BASE_DIR):

        for file in files:

            full_path = os.path.join(root, file)

            relative_path = os.path.relpath(
                full_path,
                BASE_DIR
            )

            zipf.write(
                full_path,
                arcname=relative_path
            )


print("\n==============================================")
print("DASPI OUTPUT EXPORT COMPLETE")
print("==============================================")
print("Folder:", BASE_DIR)
print("ZIP:", zip_path)
print("==============================================")


# ============================================================
# 12. GOOGLE COLAB DOWNLOAD
# ============================================================

try:

    from google.colab import files

    print("\nDownloading ZIP package...")
    files.download(zip_path)

except Exception:

    print(
        "\nIf you are not running Google Colab, "
        "the ZIP is available at:"
    )

    print(zip_path)

    # ============================================================
# SAVE ALL GENERATED GRAPHS
# ============================================================

import os
import matplotlib.pyplot as plt
from google.colab import files
import shutil

# Create folder
output_folder = "/content/DASPI_Graphs"
os.makedirs(output_folder, exist_ok=True)

# Find all currently open matplotlib figures
figure_numbers = plt.get_fignums()

print(f"Found {len(figure_numbers)} graphs.")

# Save each graph
for i, fig_num in enumerate(figure_numbers, start=1):

    fig = plt.figure(fig_num)

    filename = os.path.join(
        output_folder,
        f"DASPI_Figure_{i:02d}.png"
    )

    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    print(f"Saved: {filename}")

print("\nAll graphs saved successfully!")

# ============================================================
# SAVE AND DOWNLOAD EACH GRAPH SEPARATELY
# ============================================================

import os
import matplotlib.pyplot as plt
from google.colab import files

# Folder to temporarily store the graphs
output_folder = "/content/DASPI_Graphs"
os.makedirs(output_folder, exist_ok=True)

# Get all currently generated Matplotlib figures
figure_numbers = plt.get_fignums()

print(f"Found {len(figure_numbers)} graphs.\n")

# Save and download each graph separately
for i, fig_num in enumerate(figure_numbers, start=1):

    fig = plt.figure(fig_num)

    filename = f"DASPI_Figure_{i:02d}.png"
    filepath = os.path.join(output_folder, filename)

    # Save at high resolution for research paper
    fig.savefig(
        filepath,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"✓ Saved: {filename}")

    # Download this graph individually
    files.download(filepath)

print("\n==========================================")
print("ALL GRAPHS DOWNLOADED SEPARATELY")
print("==========================================")

In [ ]:
# ================================================================
# SPATIAL ABLATION EXPERIMENT — SYNTHETIC DEMONSTRATION ONLY
# ================================================================
# This experiment tests whether spatial constraints are connected to the
# optimisation decision. It uses synthetic eligibility and connection-cost
# assumptions because validated candidate-site and interconnection data for
# Jabalpur have not been supplied. Do not report this as empirical geography.
# Run A: aggregate/coordinate-aware model without spatial restrictions.
# Run B: synthetic spatial eligibility restrictions, capacity bounds,
# distance-based connection-cost proxy, and an aggregate grid-import cap. Both runs use the same synthetic 250 USD/MWh grid-price stress assumption so the
# storage decision is sensitive enough to test the spatial mechanism. To keep
# the ablation tractable, it uses a fixed 168-step slice (28 days at 4-hour
# resolution), not the full-year headline case. This is a synthetic method test.

import os, json, numpy as np, pandas as pd

SPATIAL_OUT = os.path.join(OUTPUT_DIR, "results", "spatial_ablation")
os.makedirs(SPATIAL_OUT, exist_ok=True)

# Deliberately explicit demonstration assumptions, to be replaced with
# sourced site-screening and interconnection data before a real-world claim.
SPATIAL_DEMO_CONFIG = {
    "eligible": {
        "Li-ion": True,
        "Flow Battery": True,
        "Pumped Hydro": False,
        "Hydrogen": False,
        "Thermal Storage": True
    },
    "connection_cost_multiplier": {
        "Li-ion": 1.15,
        "Flow Battery": 1.20,
        "Pumped Hydro": 1.00,
        "Hydrogen": 1.00,
        "Thermal Storage": 1.10
    },
    "distance_km": {
        "Li-ion": 5.0,
        "Flow Battery": 12.0,
        "Pumped Hydro": 35.0,
        "Hydrogen": 20.0,
        "Thermal Storage": 8.0
    },
    "connection_cost_usd_per_mw_km": 1000.0,
    "max_power_MW": {
        "Li-ion": 1000.0,
        "Flow Battery": 800.0,
        "Pumped Hydro": 0.0,
        "Hydrogen": 0.0,
        "Thermal Storage": 500.0
    },
    "max_energy_MWh": {
        "Li-ion": 8000.0,
        "Flow Battery": 12000.0,
        "Pumped Hydro": 0.0,
        "Hydrogen": 0.0,
        "Thermal Storage": 6000.0
    },
    "grid_import_limit_MW": 350.0,
    "assumption_status": "synthetic demonstration; not observed Jabalpur data",
    "eligible_site_logic": "technology-level hard eligibility and capacity proxy, not mapped candidate-site GIS",
    "connection_cost_logic": "synthetic distance_km x USD/MW-km plus equivalent capex multiplier; not a sourced tariff",
    "grid_connection_logic": "synthetic aggregate grid-import limit; not a network topology or power-flow model",
    "grid_cost_override": 250.0,
    "grid_cost_status": "synthetic 250 USD/MWh stress assumption applied identically to A and B; not a sourced tariff",
    "temporal_window": "first 168 model steps (28 days at 4-hour resolution); not full-year"
}

# Use the first 168 model steps (28 days at 4-hour resolution) so two
# optimisation runs are computationally tractable. Both runs use the exact
# same time slice and grid-cost stress assumption.
spatial_demo_df = df.iloc[:min(168, len(df))].copy()
if len(spatial_demo_df) < 24:
    raise ValueError("Spatial ablation requires at least 24 model time steps.")
print(f"Spatial ablation window: {len(spatial_demo_df)} steps x {RESAMPLE_HOURS} h = {len(spatial_demo_df)*RESAMPLE_HOURS} hours")

# The baseline solve uses the same inputs with no spatial restrictions.
print("\nRunning spatial ablation A: no spatial constraints")
spatial_A_result, spatial_A_portfolio, spatial_A_dispatch = build_and_solve_DASPI(
    spatial_demo_df, TECHNOLOGIES, STORAGE, grid_cost_override=SPATIAL_DEMO_CONFIG["grid_cost_override"]
)

print("\nRunning spatial ablation B: synthetic eligibility and connection-cost proxy")
spatial_B_result, spatial_B_portfolio, spatial_B_dispatch = build_and_solve_DASPI(
    spatial_demo_df, TECHNOLOGIES, STORAGE, spatial_config=SPATIAL_DEMO_CONFIG,
    grid_cost_override=SPATIAL_DEMO_CONFIG["grid_cost_override"]
)

def _spatial_summary(label, result, portfolio, dispatch):
    dt_hours = float(RESAMPLE_HOURS)
    unmet = float(dispatch["unmet_MW"].sum() * dt_hours)
    grid = float(dispatch["grid_MW"].sum() * dt_hours)
    curtail = float(dispatch["curtailment_MW"].sum() * dt_hours)
    return {
        "Run": label,
        "Solver_Success": bool(result.success),
        "Solver_Status": str(result.message),
        "Objective_Value_model_units": float(result.fun),
        "Storage_Power_MW": float(portfolio["Power_MW"].sum()),
        "Storage_Energy_MWh": float(portfolio["Energy_MWh"].sum()),
        "Grid_Imports_MWh": grid,
        "Unserved_Energy_MWh": unmet,
        "Curtailment_MWh": curtail,
        "Interpretation": "synthetic spatial ablation; not validated regional performance"
    }

spatial_ablation_summary = pd.DataFrame([
    _spatial_summary("A_no_spatial_constraints", spatial_A_result, spatial_A_portfolio, spatial_A_dispatch),
    _spatial_summary("B_synthetic_spatial_constraints", spatial_B_result, spatial_B_portfolio, spatial_B_dispatch)
])
spatial_portfolio = pd.concat([
    spatial_A_portfolio.assign(Run="A_no_spatial_constraints"),
    spatial_B_portfolio.assign(Run="B_synthetic_spatial_constraints")
], ignore_index=True)
spatial_ablation_summary.to_csv(os.path.join(SPATIAL_OUT, "spatial_ablation_summary.csv"), index=False)
spatial_portfolio.to_csv(os.path.join(SPATIAL_OUT, "spatial_ablation_portfolios.csv"), index=False)
with open(os.path.join(SPATIAL_OUT, "spatial_demo_assumptions.json"), "w", encoding="utf-8") as f:
    json.dump(SPATIAL_DEMO_CONFIG, f, indent=2)

print("\nSPATIAL ABLATION SUMMARY")
print(spatial_ablation_summary.to_string(index=False))
print("\nPortfolio comparison")
print(spatial_portfolio.to_string(index=False))
print("\nSaved spatial ablation outputs to:", SPATIAL_OUT)
print("NOTE: this tests synthetic eligibility/capacity/distance-cost/grid-import constraints only. It is NOT a network-topology or power-flow model. Replace assumptions with traceable GIS/network data before making a Jabalpur-specific claim.")


## Reviewer experiment harness and evidence boundary

Run this cell after the main model and spatial-ablation cell complete. It independently optimises No Storage, Li-ion Only, Hydrogen-only LDES, Fixed Hybrid, and DASPI-C on the same time series and saves dispatch traces and a summary table. The default is the full available horizon; set `BASELINE_MAX_STEPS = 168` only for a debugging smoke test, never as an annual result.

The code includes a synthetic spatial demonstration with hard technology eligibility/capacity limits, distance-based connection-cost proxies, and an aggregate grid-import limit. It does **not** implement GIS candidate-site allocation or full transmission topology/power-flow constraints. Those require real, sourced site/network inputs and further model development.

If empirical demand/hydropower CSVs are unavailable, the code retains explicitly labelled constructed profiles. Code changes alone cannot convert these into empirical Jabalpur observations.

In [ ]:
# ================================================================
# REVIEWER EXPERIMENT HARNESS: INDEPENDENT OPTIMISED BASELINES
# ================================================================
# Run this cell after the main model and spatial-ablation cell.
# Publication mode uses the complete available time series. Set
# BASELINE_MAX_STEPS=168 ONLY for a short smoke test; do not report that
# short-window output as an annual result.
import os, json, time, hashlib
from datetime import datetime, timezone

RUN_REVIEWER_BASELINES = True
BASELINE_MAX_STEPS = None  # None = full horizon; 168 = debugging only
BASELINE_OUT = os.path.join(RESULTS_DIR, "independent_baselines")
os.makedirs(BASELINE_OUT, exist_ok=True)

if RUN_REVIEWER_BASELINES:
    baseline_data = df.copy()
    if BASELINE_MAX_STEPS is not None:
        baseline_data = baseline_data.iloc[:int(BASELINE_MAX_STEPS)].copy()
    if len(baseline_data) < 2:
        raise ValueError("Baseline experiment needs at least two time steps.")

    baseline_cases = [
        ("No Storage", [], {}, None),
        ("Li-ion Only", ["Li-ion"], {"Li-ion": STORAGE["Li-ion"]}, None),
        ("Single LDES (Hydrogen-only)", ["Hydrogen"], {"Hydrogen": STORAGE["Hydrogen"]}, None),
        ("Fixed Hybrid (Li-ion/Hydrogen 50:50 energy)", ["Li-ion", "Hydrogen"],
         {"Li-ion": STORAGE["Li-ion"], "Hydrogen": STORAGE["Hydrogen"]},
         {"Li-ion": 0.5, "Hydrogen": 0.5}),
        ("DASPI-C", TECHNOLOGIES, STORAGE, None)
    ]

    baseline_rows = []
    baseline_portfolios = []
    baseline_run_metadata = {
        "run_started_utc": datetime.now(timezone.utc).isoformat(),
        "location_name": LOCATION_NAME,
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date_config": START_DATE,
        "end_date_config": END_DATE,
        "time_step_hours": RESAMPLE_HOURS,
        "n_steps": int(len(baseline_data)),
        "modeled_hours": int(len(baseline_data) * RESAMPLE_HOURS),
        "baseline_max_steps": BASELINE_MAX_STEPS,
        "demand_data_status": DEMAND_DATA_STATUS,
        "hydro_data_status": HYDRO_DATA_STATUS,
        "model_lifetime_years": MODEL_LIFETIME_YEARS,
        "grid_cost_usd_per_mwh": 150.0,
        "solver": "scipy.optimize.milp / HiGHS backend; record runtime version in environment",
        "note": "All cases use the same time series, objective, grid rules, horizon, and storage assumptions; results are scenario outputs, not validated Jabalpur estimates."
    }

    no_storage_dispatch = None
    for case_name, case_techs, case_params, fixed_shares in baseline_cases:
        print("\n" + "="*72)
        print("INDEPENDENT OPTIMISATION:", case_name)
        print("="*72)
        started = time.time()
        try:
            solved, case_portfolio, case_dispatch = build_and_solve_DASPI(
                baseline_data,
                case_techs,
                case_params,
                fixed_energy_shares=fixed_shares
            )
            runtime_seconds = time.time() - started
            dt_h = float(RESAMPLE_HOURS)
            load_mwh = float(case_dispatch["demand_MW"].sum() * dt_h)
            unserved_mwh = float(case_dispatch["unmet_MW"].sum() * dt_h)
            grid_mwh = float(case_dispatch["grid_MW"].sum() * dt_h)
            curtail_mwh = float(case_dispatch["curtailment_MW"].sum() * dt_h)
            renewable_mwh = float(case_dispatch["renewable_available_MW"].sum() * dt_h)
            annualized_cost_usd = (
                float(case_portfolio["Annualised_Capex_USD"].sum())
                if len(case_portfolio) and "Annualised_Capex_USD" in case_portfolio.columns else 0.0
            )
            if case_name == "No Storage":
                no_storage_dispatch = case_dispatch.copy()
            reliability_pct = 100.0 * (1.0 - unserved_mwh / load_mwh) if load_mwh > 0 else np.nan
            baseline_rows.append({
                "Scenario": case_name,
                "Solver_Status": str(solved.message),
                "Solver_Success": bool(solved.success),
                "Objective_Value_USD_equivalent": float(solved.fun),
                "Objective_Interpretation": "annualised storage capex + grid purchases + unserved-energy penalty + throughput cost, over modeled horizon; inspect objective scaling before economic interpretation",
                "Runtime_seconds": runtime_seconds,
                "Power_MW": float(case_portfolio["Power_MW"].sum()) if len(case_portfolio) else 0.0,
                "Energy_MWh": float(case_portfolio["Energy_MWh"].sum()) if len(case_portfolio) else 0.0,
                "Annualised_Storage_CAPEX_USD": annualized_cost_usd,
                "Grid_Imports_MWh": grid_mwh,
                "Unserved_Energy_MWh": unserved_mwh,
                "Curtailment_MWh": curtail_mwh,
                "Renewable_Available_MWh": renewable_mwh,
                "Load_MWh": load_mwh,
                "Reliability_percent_grid_supported": reliability_pct,
                "Interpretation": "optimised scenario on shared modeled horizon; not empirical regional validation"
            })
            if len(case_portfolio):
                _p = case_portfolio.copy()
                _p.insert(0, "Scenario", case_name)
                baseline_portfolios.append(_p)
            case_dispatch.to_csv(os.path.join(BASELINE_OUT, case_name.lower().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "").replace(":", "") + "_dispatch.csv"))
            if case_name != "No Storage" and no_storage_dispatch is not None:
                avoided_unserved_mwh = float(
                    no_storage_dispatch["unmet_MW"].sum() * dt_h - unserved_mwh
                )
                cost_musd = annualized_cost_usd / 1e6
                rasv = avoided_unserved_mwh / cost_musd if cost_musd > 0 else np.nan
                baseline_rows[-1]["Avoided_Unserved_Energy_vs_NoStorage_MWh"] = avoided_unserved_mwh
                baseline_rows[-1]["RASV_MWh_per_MUSD"] = rasv
        except Exception as exc:
            runtime_seconds = time.time() - started
            baseline_rows.append({
                "Scenario": case_name,
                "Solver_Status": "FAILED: " + repr(exc),
                "Solver_Success": False,
                "Runtime_seconds": runtime_seconds,
                "Interpretation": "failed run; no numerical performance comparison permitted"
            })
            print("CASE FAILED:", repr(exc))

    baseline_results_df = pd.DataFrame(baseline_rows)
    baseline_results_df.to_csv(os.path.join(BASELINE_OUT, "independent_baseline_summary.csv"), index=False)
    if baseline_portfolios:
        pd.concat(baseline_portfolios, ignore_index=True).to_csv(
            os.path.join(BASELINE_OUT, "independent_baseline_portfolios.csv"), index=False
        )
    baseline_run_metadata["run_finished_utc"] = datetime.now(timezone.utc).isoformat()
    baseline_run_metadata["all_cases_successful"] = bool(
        len(baseline_results_df) == len(baseline_cases)
        and baseline_results_df["Solver_Success"].fillna(False).all()
    )
    with open(os.path.join(BASELINE_OUT, "run_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(baseline_run_metadata, f, indent=2)

    # Save event-level data and run-specific metrics for reproducibility.
    events.to_csv(os.path.join(RESULTS_DIR, "deficit_event_table.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, "metric_definitions.json"), "w", encoding="utf-8") as f:
        json.dump({
            "DMI": {
                "formula": "sum(E_k * max(d_k - 8 h, 0)) / (8 h * sum(E_k))",
                "units": "dimensionless",
                "reference_duration_hours": REFERENCE_DURATION_HOURS,
                "interpretation": "deficit-energy-weighted excess duration relative to 8 h; can exceed 1"
            },
            "TSCR": {
                "formula": TSCR_DEFINITION,
                "units": "fraction of detected events (0-1) passing the optimistic peak-power and usable-energy screen",
                "limitations": "assumes each event starts fully charged; not a chronological dispatch result"
            },
            "RASV": {
                "formula": RASV_DEFINITION,
                "units": "MWh avoided per million USD of annualised storage CAPEX over the modeled horizon",
                "limitations": "only valid where a successful No Storage case and successful comparison case are available"
            },
            "resource_conversion": {
                "solar": SOLAR_INPUT_UNIT_ASSUMPTION,
                "pv_derate": PV_SYSTEM_DERATE,
                "pv_temperature_coefficient_per_C": PV_TEMPERATURE_COEFFICIENT_PER_C,
                "wind_hub_height_m": WIND_HUB_HEIGHT_M,
                "wind_shear_exponent": WIND_SHEAR_EXPONENT,
                "wind_curve": WIND_TURBINE_MODEL
            }
        }, f, indent=2)

    print("\nINDEPENDENT BASELINE RESULTS")
    print(baseline_results_df.to_string(index=False))
    print("\nSaved baseline tables and dispatch traces to:", BASELINE_OUT)
    if not baseline_run_metadata["all_cases_successful"]:
        print("WARNING: at least one case failed; do not claim comparative superiority.")
else:
    print("Baseline runs skipped. Set RUN_REVIEWER_BASELINES=True to execute the independent optimisations.")
